## Pattern ISC - Post by Post Analysis

Given that each subject watched posts in a different order, we analyze the data post by post.
This allows us to see how brain responses vary for each specific post across subjects, across post type.

**Analysis plan inspiration from:**
> Chen, J., Leong, Y., Honey, C. et al. Shared memories reveal shared structure in neural activity across individuals. Nat Neurosci 20, 115–125 (2017). https://doi.org/10.1038/nn.4450

**Analysis Steps:**
1. Denoised BOLD data is loaded from the previous notebook.
2. Post timings are loaded from the metadata - e-prime event files for each subject, for each run.
3. Post BOLD timeseries is extracted - then averaged across TRs within each post to have one value per post per voxel per subject.

(1) per-subject/run event loading → (2) TR windows → (3) event-wise pattern extraction with a 5×5×5 searchlight

In [ ]:
"""
Full Searchlight Pipeline: All Subjects
========================================

This script runs the complete searchlight ISC analysis:
1. Extract searchlight patterns from all subjects
2. Align patterns across subjects
3. Compute ISC for each searchlight sphere
4. Statistical testing with permutations

Date: 2026-02-22
"""

from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
import importlib
from yy_fmri_kit import event_isc, isc
importlib.reload(event_isc)
importlib.reload(isc)
# Import from yy-fMRI-kit
from yy_fmri_kit.event_isc.alignment import PatternAligner
from yy_fmri_kit.event_isc.extraction import SearchlightPatternExtractor
from yy_fmri_kit.static.event_isc.config import (
    ExtractionConfig,
    AlignmentConfig,
)
from yy_fmri_kit.event_isc.extraction import (SearchlightPatternExtractor)
from yy_fmri_kit.event_isc.alignment import (PatternAligner)
from yy_fmri_kit.isc.analyzer import ISCAnalyzer

# ============================================================================
# CONFIGURATION - CHANGE THESE TO MATCH YOUR DATA
# ============================================================================

# Data paths
DATA_DIR = Path("/path/to/data/derivatives/denoised")
EVENTS_CSV = Path("/path/to/behavioral_analyses/data/combined_events_with_bids.csv")
# Output directory
OUTPUT_DIR = Path("/path/to/data/derivatives/searchlight/GM")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Experiment parameters
TR = 1.0              # Your TR in seconds
SHIFT_TR = 4          # Hemodynamic delay in TRs
SEARCHLIGHT_RADIUS = 9.0  # Radius in mm

# Run types to analyze
RUN_TYPES = ["AntiLeft", "AntiRight", "ProLeft", "ProRight"]
TEST_SUBJECTS = [
    'sub-1', 'sub-6', 'sub-20', 'sub-21', 'sub-22', 'sub-23', 
    'sub-24', 'sub-25', 'sub-26', 'sub-27', 'sub-30', 'sub-31'
]

In [ ]:
# ============================================================================
# OPTIONAL - CREATE A STANDARD MASK (if you don't have one)
# ============================================================================

from nilearn.datasets import load_mni152_brain_mask

standard_mask = load_mni152_brain_mask(resolution=2)
MASK_FILE = OUTPUT_DIR / "group_mask.nii.gz"
standard_mask.to_filename(str(MASK_FILE))
print(f"Saved mask: {MASK_FILE}, voxels: {int(standard_mask.get_fdata().sum())}")

print("Standard mask shape:", standard_mask.shape)  # should be (91, 109, 91)
print("N voxels:", int(standard_mask.get_fdata().sum()))  # should be ~130,000

In [ ]:
# ============================================================================
# OPTIONAL - CREATE A GRAY MATTER MASK (if you want to restrict to GM)
# ============================================================================
from nilearn.datasets import fetch_icbm152_2009
from nilearn import image
import nibabel as nib

# Reload GM probability map
icbm = fetch_icbm152_2009()
gm_prob = image.load_img(icbm['gm'])

# Reload BOLD reference
bold_files = list(DATA_DIR.rglob("*task-*_bold.nii.gz"))
ref_img = nib.load(str(bold_files[0]))
ref_3d = image.index_img(ref_img, 0)

# Resample and threshold
gm_resampled = image.resample_to_img(gm_prob, ref_3d, interpolation='linear')
gm_mask = image.math_img("img > 0.5", img=gm_resampled)

MASK_FILE = OUTPUT_DIR / "gm_mask_bold_space.nii.gz"
gm_mask.to_filename(str(MASK_FILE))

print("Shape:", gm_mask.shape)    # should be (91, 109, 91)
print("Voxels:", int(gm_mask.get_fdata().sum()))  # should be ~50-70k

In [ ]:
# ============================================================================
# ISC CONFIGURATION
# ============================================================================

MASK_FILE = OUTPUT_DIR / "gm_mask_bold_space.nii.gz"  # Path to group mask, or None for whole brain

# ISC parameters
N_PERMUTATIONS = 1000  # For statistical testing (1000+ recommended)
ALPHA = 0.05
CORRECTION = "fdr_bh"  # "fdr_bh", "bonferroni", or "none"

# Processing
N_JOBS = -1  # Number of parallel jobs (-1 = all CPUs)

print("="*80)
print("SEARCHLIGHT ISC PIPELINE")
print("="*80)
print(f"\nStarted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nConfiguration:")
print(f"  Data directory: {DATA_DIR}")
print(f"  Events file: {EVENTS_CSV}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  TR: {TR}s")
print(f"  Shift: {SHIFT_TR} TRs ({SHIFT_TR * TR}s)")
print(f"  Searchlight radius: {SEARCHLIGHT_RADIUS}mm")
print(f"  Run types: {RUN_TYPES}")
print(f"  Permutations: {N_PERMUTATIONS}")
print(f"  Parallel jobs: {N_JOBS}")

In [ ]:
# ============================================================================
# STEP 0: LOAD DATA AND BUILD RUNS DICTIONARY
# ============================================================================

print("\n" + "="*80)
print("STEP 0: LOADING DATA")
print("="*80)

# Load events
print(f"\nLoading events from: {EVENTS_CSV}")
events_df = pd.read_csv(EVENTS_CSV)
print(f"✓ Loaded {len(events_df)} events")
print(f"  Subjects: {events_df['subject'].nunique()}")
print(f"  Run types: {events_df['run_type'].unique().tolist() if 'run_type' in events_df.columns else 'N/A'}")

# Build runs dictionary
print("\nBuilding runs dictionary...")
runs_dict = {}

# Find subject directories
subject_dirs = sorted([d for d in DATA_DIR.glob("sub-*") if d.is_dir()])
print(f"Found {len(subject_dirs)} subject directories")

for sub in TEST_SUBJECTS:  # Limit to first 3 subjects for testing
    subject_dir = DATA_DIR / sub
    
    # Find all NIfTI files for this subject
    nifti_files = list(subject_dir.rglob("*_bold.nii.gz"))
    
    # Filter for task runs (adjust pattern to match your files)
    task_files = [
        f for f in nifti_files 
        if any(task in f.name.lower() for task in ['antileft', 'antiright', 'proleft', 'proright'])
    ]
    
    if task_files:
        runs_dict[sub] = task_files
        print(f"  {sub}: {len(task_files)} runs")

print(f"\n✓ Built runs dictionary with {len(runs_dict)} subjects")
total_runs = sum(len(files) for files in runs_dict.values())
print(f"  Total runs: {total_runs}")

In [ ]:
# ============================================================================
# STEP 1: EXTRACT SEARCHLIGHT PATTERNS
# ============================================================================

print("\n" + "="*80)
print("STEP 1: EXTRACTING SEARCHLIGHT PATTERNS")
print("="*80)

# Configure extraction
extraction_config = ExtractionConfig(
    tr=TR,
    shift_tr=SHIFT_TR,
    time_unit="seconds",
    smoothing_fwhm=None,  # No smoothing for searchlight (done at searchlight level)
    detrend=True,
    standardize=False,    # ISC computation handles standardization
    searchlight_radius=SEARCHLIGHT_RADIUS,
    searchlight_n_jobs=N_JOBS,
    valid_run_types=RUN_TYPES,
    # IMPORTANT: Set your actual column names here
    post_col="post_id",  # ← CHANGE to your post ID column
    subject_col="bids_id",
    run_col="run",
)

print("\nExtraction configuration:")
print(f"  TR: {extraction_config.tr}s")
print(f"  Hemodynamic shift: {extraction_config.shift_tr} TRs")
print(f"  Searchlight radius: {extraction_config.searchlight_radius}mm")
print(f"  Post column: '{extraction_config.post_col}'")
print(f"  Subject column: '{extraction_config.subject_col}'")
print(f"  Run column: '{extraction_config.run_col}'")

# Create extractor
extractor = SearchlightPatternExtractor(extraction_config)

# Output directory for extracted patterns
extraction_output = OUTPUT_DIR / "searchlight_patterns"

print(f"\nExtracting patterns...")
print(f"  Output: {extraction_output}")
print(f"  This may take a while (searchlight is slower than ROI)...")

# Run batch extraction
summary = extractor.batch_extract(
    runs_dict=runs_dict,
    events_df=events_df,
    output_dir=extraction_output,
    mask_path=MASK_FILE,
    verbose=False,  # Set True for detailed logging
)

print("\n✓ Extraction complete!")
print("\nExtraction summary:")
print(summary.groupby('status').size())

# Save summary
summary_file = OUTPUT_DIR / "extraction_summary.csv"
summary.to_csv(summary_file, index=False)
print(f"\nSaved summary to: {summary_file}")

# Show successful extractions
success = summary[summary['status'] == 'success']
if len(success) > 0:
    print(f"\nSuccessful extractions:")
    print(f"  Runs: {len(success)}")
    print(f"  Mean posts/run: {success['n_posts'].mean():.1f}")
    print(f"  Mean searchlights: {success['n_searchlights'].mean():.0f}")

# Show failures
failures = summary[summary['status'] != 'success']
if len(failures) > 0:
    print(f"\n⚠️  {len(failures)} runs failed:")
    print(failures[['subject', 'run_type', 'status']])


In [ ]:
# ============================================================================
# STEP 2: ALIGN PATTERNS ACROSS SUBJECTS
# ============================================================================

print("\n" + "="*80)
print("STEP 2: ALIGNING PATTERNS ACROSS SUBJECTS")
print("="*80)

# Configure alignment
alignment_config = AlignmentConfig(
    strategy="intersection",  # Only posts seen by ALL subjects
    min_subjects=2,
    allow_missing_posts=False,
    check_feature_dims=True,
)

print("\nAlignment configuration:")
print(f"  Strategy: {alignment_config.strategy}")
print(f"  Min subjects: {alignment_config.min_subjects}")

# Create aligner
aligner = PatternAligner(alignment_config)

# Align all run types
print("\nAligning run types...")
aligned_results = aligner.align_all_runs(
    output_dir=extraction_output,
    run_types=RUN_TYPES,
    pattern_type="searchlight"
)

print(f"\n✓ Aligned {len(aligned_results)} run types")

# Save aligned data
aligned_dir = OUTPUT_DIR / "aligned"
aligned_dir.mkdir(exist_ok=True, parents=True)

for run_type, result in aligned_results.items():
    output_file = aligned_dir / f"{run_type}_searchlight_aligned.npz"
    aligner.save_aligned(result, output_file, run_type)
    
    print(f"\n{run_type}:")
    print(f"  Subjects: {len(result['subjects'])}")
    print(f"  Posts: {len(result['post_ids'])}")
    print(f"  Searchlights: {result['data'][0].shape[1]}")
    print(f"  Shape per subject: {result['data'][0].shape}")
    print(f"  Saved: {output_file.name}")

In [ ]:
# ============================================================================
# STEP 3: COMPUTE ISC FOR EACH RUN TYPE
# ============================================================================

print("\n" + "="*80)
print("STEP 3: COMPUTING ISC WITH PERMUTATION TESTING")
print("="*80)

# Configure ISC analysis
analyzer = ISCAnalyzer(
    backend="native",  # Use your compute_isc
    fisher_z=True,     # Recommended for inference
    nan_policy="omit", # Handle missing data
)

print("\nISC configuration:")
print(f"  Backend: {analyzer.backend}")
print(f"  Fisher z: {analyzer.fisher_z}")
print(f"  Permutations: {N_PERMUTATIONS}")
print(f"  Alpha: {ALPHA}")
print(f"  Correction: {CORRECTION}")

# Analyze each run type
isc_results = {}
isc_output = OUTPUT_DIR / "isc_results"
isc_output.mkdir(exist_ok=True, parents=True)

for run_type in aligned_results.keys():
    print(f"\n{'#'*80}")
    print(f"Analyzing: {run_type}")
    print(f"{'#'*80}")
    
    # Load aligned data
    aligned_file = aligned_dir / f"{run_type}_searchlight_aligned.npz"
    
    # Run ISC analysis
    results = analyzer.analyze_from_file(
        aligned_file,
        n_permutations=N_PERMUTATIONS,
        alpha=ALPHA,
        correction=CORRECTION,
        random_seed=42,
    )
    
    isc_results[run_type] = results
    
    # Save results
    results_file = isc_output / f"{run_type}_searchlight_isc.npz"
    analyzer.save_results(results, results_file)
    print(f"\nSaved results to: {results_file}")

In [ ]:
# ============================================================================
# STEP 4: SUMMARY AND COMPARISON
# ============================================================================

print("\n" + "="*80)
print("STEP 4: SUMMARY")
print("="*80)

# Compare across run types
comparison_data = []
for run_type, results in isc_results.items():
    comparison_data.append({
        'run_type': run_type,
        'n_subjects': results['isc_subjectwise'].shape[0],
        'n_posts': results['isc_subjectwise'].shape[1] if results['isc_subjectwise'].ndim > 2 else 1,
        'n_searchlights': len(results['isc_mean']),
        'mean_isc': results['isc_mean'].mean(),
        'std_isc': results['isc_mean'].std(),
        'n_significant': results['n_significant'],
        'pct_significant': 100 * results['n_significant'] / len(results['significant']),
        'mean_isc_sig': results['isc_mean'][results['significant']].mean() if results['n_significant'] > 0 else np.nan,
    })

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*80)
print("ISC COMPARISON ACROSS RUN TYPES")
print("="*80)
print(comparison_df.to_string(index=False))

# Save comparison
comparison_file = OUTPUT_DIR / 'isc_comparison.csv'
comparison_df.to_csv(comparison_file, index=False)
print(f"\n✓ Saved comparison to: {comparison_file}")

In [ ]:
# ============================================================================
# STEP 5: SAVE RESULTS AS NIfTI MAPS FOR VISUALIZATION
# ============================================================================
import nibabel as nib
import numpy as np
from nilearn import image

# Load your ISC results
for run_type, results in isc_results.items():
    
    # Load searchlight centers and affine from one extraction file
    sample_file = list((OUTPUT_DIR / "searchlight_patterns").rglob("*.npz"))[0]
    with np.load(sample_file, allow_pickle=True) as meta:
        affine = meta['affine']
        centers = meta['searchlight_centers']
    
    # Get the brain mask shape
    group_mask = nib.load(str(MASK_FILE))
    
    # --- Map 1: Mean ISC ---
    isc_map = np.zeros(group_mask.shape)
    for i in range(len(results['isc_mean'])):
        x, y, z = centers[i].astype(int)
        isc_map[x, y, z] = results['isc_mean'][i]
    
    isc_nii = nib.Nifti1Image(isc_map, affine)
    isc_nii.to_filename(str(OUTPUT_DIR / f"{run_type}_isc_mean.nii.gz"))
    
    # --- Map 2: Significance mask (1 = significant, 0 = not) ---
    sig_map = np.zeros(group_mask.shape)
    for i in range(len(results['significant'])):
        x, y, z = centers[i].astype(int)
        sig_map[x, y, z] = float(results['significant'][i])
    
    sig_nii = nib.Nifti1Image(sig_map, affine)
    sig_nii.to_filename(str(OUTPUT_DIR / f"{run_type}_significant_mask.nii.gz"))
    
    # --- Map 3: Thresholded ISC (only significant voxels) ---
    thresh_map = isc_map * sig_map
    thresh_nii = nib.Nifti1Image(thresh_map, affine)
    thresh_nii.to_filename(str(OUTPUT_DIR / f"{run_type}_isc_thresholded.nii.gz"))
     
    print(f"{run_type}: saved 3 NIfTI files")
    print(f"  ISC range: {results['isc_mean'].min():.3f} to {results['isc_mean'].max():.3f}")
    print(f"  Significant voxels: {results['significant'].sum()}")

In [ ]:
# ============================================================================
# STEP 6: VISUALIZE RESULTS (OPTIONAL)
# ============================================================================

from nilearn import plotting
import nibabel as nib
import numpy as np

run_type = "ProLeft"  # Change to your run type

# Load your thresholded ISC map
isc_nii = nib.load(str(OUTPUT_DIR / f"{run_type}_isc_thresholded.nii.gz"))

# --- 1. Glass brain (good overview) ---
plotting.plot_glass_brain(
    isc_nii,
    colorbar=True,
    title=f"ISC - {run_type}",
    cmap="hot",
    vmin=0.05,
    vmax=0.4,
    plot_abs=False,
    display_mode="lyrz",
)
plotting.show()

# --- 2. Stat map on MNI (more anatomical) ---
plotting.plot_stat_map(
    isc_nii,
    colorbar=True,
    title=f"ISC - {run_type}",
    cmap="hot",
    vmax=0.4,
    threshold=0.05,  # only show voxels above this ISC value
    display_mode="z",
    cut_coords=10,   # number of axial slices
)
plotting.show()

# --- 3. Interactive HTML (great for exploration) ---
html = plotting.view_img(
    isc_nii,
    bg_img="MNI152",
    cmap="hot",
    threshold=0.05,
    title=f"ISC - {run_type}",
    vmax=0.4,
)
html.save_as_html(str(OUTPUT_DIR / f"{run_type}_isc_interactive.html"))
print("Saved interactive map — open in browser")

# visualize significant voxels only in an interactive map
html_sig = plotting.view_img(
    str(OUTPUT_DIR / f"{run_type}_isc_thresholded.nii.gz"),  # ISC values, non-sig = 0
    bg_img="MNI152",
    cmap="hot",
    threshold=0.01,  # just above zero to exclude non-significant voxels
    title=f"Significant ISC - {run_type}",
    vmin=0.1,
    vmax=0.5,
)
html_sig.save_as_html(str(OUTPUT_DIR / f"{run_type}_significant_interactive.html"))
print(f"Saved: {run_type}_significant_interactive.html — open in browser")
